<h1><center>🧠 Homework №1 — BERT for Aspect-Based Sentiment Classification

*Natural Language Processing Course, HSE 2026*

This homework is designed as a hands-on, research-style assignment. You
are expected not only to train models, but also to make methodological
decisions, justify them, and analyze results critically (*this will be taken into account when grading your assignment*) — like an ML
engineer or NLP researcher.

This homework **is based on the materials from the Webinar 2** - take a look at the notebook from that class as it might be really helpful when completing this assignment.

<p align="center">
  <img src="https://www.philschmid.de/static/blog/bert-text-classification-in-a-different-language/meme.png" width="60%">
</p>

### 📥 General Rules and Submission Guidelines

1. Copying code from external sources (**including using LLMs**) without explicit citation is strictly prohibited and will result in 0 points for the entire assignment. If you consult any resources or AI tools, you must clearly state this in a separate Markdown cell. If suspected of this, you might be asked to explain your code to the grader and answer their questions during a separate session to avoid the mentioned penalty.
2. All results must be fully **reproducible**. You are required to use `set_seed` everywhere so that the grader can obtain the same results when rerunning your notebook.
3. The notebook must run from top to bottom without errors. Submissions that fail to execute sequentially will not be accepted.
4. You must satisfy all requirements in each task to receive full credit. Partial completion may lead to partial scoring.
5. Do not modify the original notebook structure or provided Markdown cells. You are only allowed to write code in the sections marked `# TODO: your code here`. Any explanations, interpretations, or additional comments must be placed **in separate Markdown cells**. If you choose to do so, leave an explanation as to what and why was changed.
6. The final submission must be a completed `.ipynb` Jupyter Notebook. You may conduct your experiments in Jupyter Notebook, VS Code, or Google Colab — whichever environment you prefer.

### 🎯 Learning Objectives

By completing this homework, you will:
- Work with a realistic, noisy text dataset.
- Perform careful data preprocessing and label engineering.
- Fine-tune multiple encoder-based transformer models.
- Use validation loss for model selection (early stopping).
- Conduct basic hyperparameter tuning.
- Log your results.
- Build ablation tables and compare experimental settings.
- Interpret results obtained through conducted experiments.

Without any further ado, let's get started. Good luck, and may the odds be ever in your favor!

### 🔧 Environment Setup

Loading necessary libraries. If you need anything else, feel free to add more libraries and dependencies.

In [ ]:
! pip install transformers[torch] datasets evaluate optuna -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)

from datasets import Dataset
import evaluate
import optuna
import random


# TODO: your code here

In [ ]:
if not torch.cuda.is_available():
    print("Warning: training will be slow without GPU!")

In [ ]:
def set_seed(seed):
    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(1337)

### 🗂️ **Part №1. Data Preparation. ($1$ point)**

In this assignment, we're going to be work with the dataset containing reviews on restaurants. The dataset can be downloaded from [here](https://drive.google.com/file/d/19qXNsW-gc_H4ufAt0JREzszKzjKkh170/view).

In this part of the assignment, you will prepare the raw restaurant reviews dataset for model training. The goal is to obtain clean text data, a well-defined target variable, and reproducible train/validation/test splits.

**1.1. Loading and inspecting data ($0.25$ points)**

First, create a code cell that:
- loads the .jsonl file,
- prints basic statistics (size, columns, example rows),
- briefly comments on what you see.

In [ ]:
!wget -O restaurants_reviews.jsonl \
"https://drive.google.com/uc?export=download&id=1EtlYDa05wGaF_6kqdb08XbKMRTliSlrT"

--2026-02-06 07:13:33--  https://drive.google.com/uc?export=download&id=1EtlYDa05wGaF_6kqdb08XbKMRTliSlrT
Resolving drive.google.com (drive.google.com)... 173.194.216.100, 173.194.216.139, 173.194.216.102, ...
Connecting to drive.google.com (drive.google.com)|173.194.216.100|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1EtlYDa05wGaF_6kqdb08XbKMRTliSlrT&export=download [following]
--2026-02-06 07:13:33--  https://drive.usercontent.google.com/download?id=1EtlYDa05wGaF_6kqdb08XbKMRTliSlrT&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 172.217.204.132, 2607:f8b0:400c:c15::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|172.217.204.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 80131991 (76M) [application/octet-stream]
Saving to: ‘restaurants_reviews.jsonl’

restaurants_reviews 100%[===================>] 

In [ ]:
df = # TODO: load jsonl into a DataFrame
df.head(3)

,review_id,general,food,interior,service,text
0,0,0,10,10,10,Вытянули меня сегодня в город и раз уж была в ...
1,1,0,9,10,9,проводили корпоратив на 60 чел. в этот - уже т...
2,2,0,9,10,10,Был в Гостях с женой один раз и еще раз с жено...


In [ ]:
# TODO: print out statistics and add your comments as an md cell (what we're seeing, any trends - whatever seems relevant to the observed stats)

**1.2 Target variable selection and cleaning ($0.25$ points)**

#### 📚 Quick Dataset Overview

As you have probably already guessed, you will work with a dataset of Russian restaurant reviews stored in restaurants_reviews.jsonl. Each row corresponds to one user review and includes:

- review_id — unique identifier of the review  
- text — the review text (input to the model)  
- food — rating of food quality  
- service — rating of service quality  
- interior — rating of interior/atmosphere  
- general — overall rating (summary score)

The key idea is that the dataset provides aspect-based ratings: the same review can express different opinions about food, service, and interior. For example, a review might praise the food but criticize the service. This makes the task more realistic than standard “overall sentiment” classification.

Now, your task is to choose ONE aspect as the target variable for sentiment classification:

- ✅ food  
- ✅ service  
- ✅ interior  
- ❌ general cannot be used as a target label in this homework.

After that,
- Once you've picked your target - keep only 2 columns (target variable - name it "`food/interior/service_score`"  and `text`). You will not need id or other aspects (the ones you did NOT choose) for the training process, so we'll remove them for convenience.
- After selecting your target aspect, remove all samples where the corresponding rating equals $0$.  
In this dataset, a score of $0$ indicates that the reviewer **did not provide a rating for this aspect**, so such reviews should not be included in further analysis.
- Explain in a Markdown comment why the previous step is necessary (removal of $0$-scored reviews).

The result of this task is a df with 2 columns, e.g. "interior_score", "text" with no missing values.

In [ ]:
# TODO: your code here

In [ ]:
# TODO: your code here

In [ ]:
# TODO: your code here

**1.3 Label binning into 3 classes ($0.25$ points)**

At this point, you have a dataset that contains raw scores for your chosen aspect in the range 1–10 (after removing missing values).

Your goal in this section is to reduce this 10-point scale to three sentiment classes:

 - $0$ → BAD
 - $1$ → NEUTRAL
 - $2$ → GOOD

Before creating these classes, you must first visualize the original rating distribution so that your binning strategy is data-driven rather than arbitrary.
In the next cell, plot the distribution of the raw ratings using either:
 - a histogram, or
 - a bar/count plot.

In [ ]:
# TODO: visualize distribution

plt.figure(figsize=(10, 6))

# TODO: replace ___ with correct parameters
sns._____(data=df, x="______", ______б palette="viridis")

plt.title("Distribution of Original Ratings")
plt.xlabel("Rating")
plt.ylabel("Count")
plt.______()

After inspecting the plot, add a Markdown explanation describing how and why you have decided to map the 1–10 scale into three groups.

**Your explanation here:** ...

Now, implement your mapping in code into three sentiment classes: BAD ($0$), NEUTRAL ($1$), and GOOD ($2$). Your mapping must produce labels in {$0$, $1$, $2$}.

In [ ]:
# TODO: mapping
def bin_label(rating):
  pass

In [ ]:
# TODO: print out results

**1.4. Formatting ($0.25$ points)**

At this point in the pipeline, we already have:
- a cleaned dataset of reviews, and
- their corresponding sentiment labels in {0, 1, 2}.

However, our data is still stored in a pandas DataFrame, which is not the native format expected by the Hugging Face training ecosystem.

**Task:** to make our data compatible with the Trainer API and other transformer utilities, you need to
- convert it into a datasets.Dataset object
- perform train/val/test split. The proprotions are 70/15/15

In [ ]:
# TODO: your code here

In [ ]:
# TODO: your code here

In [ ]:
data

### 💻 **Transition Part ($1$ ⏩ $2$). From text to tensors. ($0.5$ points)**

Some theory for a better understanding of **why** (*but why?*... 🐧) we're doing **what** we're doing:

At this stage of the pipeline, we already have clean texts and well-defined labels in the right format, but transformer models cannot operate directly on raw strings. They require numerical representations that reflect how each specific model was trained. This is why we perform tokenization before training.

It is important to emphasize that tokenization is model-dependent. Different pretrained models were trained on different corpora and use different vocabularies, token boundaries, and subword segmentation rules. As a result, the same sentence may be split into different tokens by different models, even though the underlying meaning is identical. This is why we cannot tokenize the data once and reuse it for all models.

**In this assignment, the tokenization step must be repeated separately for each model.**

For each model separately, you will:
 1. Load the model-specific tokenizer from Hugging Face.
 2. Tokenize all three splits (train, val, test) of the Dataset object.
 3. Wrap the tokenized outputs and labels into a DataLoader, creating 3 separate dataloader objects.

Only after this step will you proceed to fine-tuning.


As our first model, let us take [ruBert-base](https://huggingface.co/sberbank-ai/ruBert-base/). Perform tokenization in the following cells.

In [ ]:
# TODO: perform data tokenization

data_tokenized

Wrap the tokenized data into a DataLoader

In [ ]:
# TODO: collator initialization and wrapping

In [ ]:
train_dataloader = ...
val_dataloader = ...
test_dataloader = ...

**What you enter Part 2 with:**

At the start of Part 2, you should have:
1. 3 Dataloader objects
2. All of them must be tokenized (you will later need to change it, depending on what model you are training),

### 💪 **Part 2. Fine-tuning. ($3$ points)**

In this part of the assignment, you will complete the following stages:
 1. **Baseline model.**
Train a baseline encoder model on the prepared dataset. This establishes a reference point for all future comparisons. ($1$ point)
 2. **Model comparison (3 encoders total).**
Train two additional encoder models under the same data split and evaluation protocol. Compare results across all three models and choose the most promising model to continue working with. ($2$ points)

Training must be stopped automatically via early stopping: once the quality metric stops improving (i.e., reaches a plateau), training should terminate and the best checkpoint must be selected. Do not forget that data is shared across all models (do not redo the split), but **tokenization is model-specific**.

#### Models to choose from:
1. [ruBert-base](https://huggingface.co/sberbank-ai/ruBert-base/)  
2. [ruBert-large](https://huggingface.co/sberbank-ai/ruBert-large/)
3. [ruBert-tiny](https://huggingface.co/cointegrated/rubert-tiny2)
4. [mBert](https://huggingface.co/google-bert/bert-base-multilingual-cased)

More documentation: [`transformers`](https://huggingface.co/docs/transformers/index)


Throughout this assignment, the primary metric is `accuracy` (multiclass classification).
Before training any model, you must implement a function that computes accuracy from model predictions. In the next section, you will define this metric function and integrate it into your training loop.

In [ ]:
def compute_metrics(eval_pred):
    """ Func for calculating accuracy
    (The primary metric must be accuracy. You may optionally add others) """

    logits, labels = ... # TODO
    preds = np.argmax(logits, axis=-1)
    return ... # TODO

Next, choose a model (overall you need to conduct initial experiments with 3 of them). In the initial task, we already used a tokenizer for ruBert-base, so you might want to proceed with that first.

In [ ]:
# TODO: load a pretrained encoder
model_name = ...
model = AutoModelForSequenceClassification.from_pretrained(
    ...,
    num_labels=3
)

Thanks to a user-friendly interface of training, a developer can focus on high-level tasks (rather than implementing a train loop):

Modify `Trainer` ([documentation here](https://huggingface.co/docs/transformers/en/main_classes/trainer#trainer)) and `TrainingArguments`([documentation here](https://huggingface.co/docs/transformers/en/main_classes/trainer#transformers.TrainingArguments)) and complete the following code cells.


Do not forget to add `load_best_model_at_end=True`, `metric_for_best_model=...`, `greater_is_better=True` as well as `evaluation_strategy=...`

Use `EarlyStoppingCallback` to ensure training is stopped once the metric used for evaluation stops increasing over an established number of epochs.

In [ ]:
# TODO: define training settings
args_es = TrainingArguments(
    ... , # TODO
    run_name="BERT-early_stopping",
    load_best_model_at_end=True,
    logging_steps=10,
)

In [ ]:
# TODO: pass the datasets into Trainer and begin training
trainer = Trainer(
    model=model,
    args=args_es,
    ...
    ...
    ...
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
trainer.train()

Do this for 3 different encoders (choose them from what was offered previously).

In [ ]:
# TODO: training here

Now, before proceedings to hyperparamter tuning, you must compare the three pretrained encoders you trained under the **same** data split and evaluation protocol. Create an ablation table and report:
1. Model name
2. Key hyperparameter values (they must be the same for all models for the results to be comparable)
3. Number of epochs you ran
4. Best validation accuracy

In [ ]:
# TODO: your table here

$#$ TODO: ablation table


After completing the table, select a model for further hyperparameter tuning and explain your choice. Zero analysis and/or explanation will result in zero points.

**Your answer here:** ...

#### 🔧 **Part 3. Hyperparameter tuning ($3$ points)**

In this part you will perform the following:

 1. **Hyperparameter tuning** (selected model only).
After choosing the best-performing model from Part 2, perform hyperparameter tuning on `train` and `val` portions of the dataset (not on test!!!). You may do this manually or using Optuna. Your goal is to improve validation performance while keeping the experiment design fair and reproducible ($2$ points).
 2. **Final evaluation.**
Take the best checkpoint from your tuned setup and report the final model quality on the held-out `test` set. The test set must not be used for training decisions ($1$ point).

As stated earlier, you can either perform hyperparameter tuning manually  or you can use `Optuna`. The `Trainer` API includes built-in integration with `Optuna`, which is a tool for hyperparameter optimization in machine learning.

The list of hyperparameters you may want to choose from:
1. Learning rate
2. Batch size
3. Max sequence length
4. Weight decay
5. Warmup ratio
6. Type of scheduler

Your task is to implement a search over at least $2$ different hyperparameters.
A detailed tutorial can be found [here](https://huggingface.co/docs/transformers/hpo_train).

It is important to keep in mind that hyperparameter search should be performed on a small subset of the training data, and evaluation metrics should be computed on a small subset of the validation data.

In [ ]:
# TODO: your code here

To take a look and access the best hyperparamters, you can refer to the corresponding attribute:

In [ ]:
best_trial.hyperparameters

Now, train your selected model with these values of hyperparameters on a full dataset and evaluate its final quality on test data.

In [ ]:
# TODO: your code here

Create a table with the final results of the training process and comment on what results you have obtained.

In [ ]:
# TODO: your final results here

**Your comments here:** ...

#### 😎 **Part 4. Final Experiments and Analysis ($2.5$ points)**

**4.1. Layer Freezing Experiment ($1.25$ points)**
Choose one of the four pretrained models from earlier.
You must:
 1. Freeze all encoder layers except the last transformer block and the classification head.
 2. Train the model on the prepared data.
 3. Report test accuracy for the best checkpoint.
 4. Analyse your results and comment on them in a separate md cell. No analysis or/and comments on your results will result in 0 points for this task.

(*See example in the notebook from the BERT webinar*)

In [ ]:
# TODO: your code here

**Your comments here:** ...

**4.2. Return to the original 10-class problem ($1.25$ points)**

Take the best model configuration from Part 3 (after tuning).

You must:
 1. Rebuild the dataset for the same selected aspect as in Part 1, but now use its original raw ratings ($1$–$10$) as labels.  
   That is, keep **the same target aspect**, but omit the BAD/NEUTRAL/GOOD binning step into $3$ classes ($0$, $1$, $2$) and treat each distinct rating as its own class.
 2. If the train set is imbalanced, apply a balancing method.
 3. Fine-tune the model again.
 4. Evaluate the final checkpoint on the test set and report accuracy.
 5. Analyse your results and comment on them in a separate md cell. No analysis or/and comments on your results will result in 0 points for this task.


In [ ]:
# TODO: your code here

**Your comments here:** ...